![Platform+](images/banner.png)

# Lab 8: Time Travel
## Undo an accidental DROP SCHEMA — and govern retention

> **Persona track: Platform Admin** — same lab, your angle.

:point_right: **In this lesson, you'll stand up a schema, accidentally drop the whole thing during "cleanup", recover it in one statement with `UNDROP SCHEMA`, then take the admin's view: tune the Time Travel retention window and understand the Time Travel / Fail-safe boundary you're accountable for.**

The scenario: You're the Platform Admin at **Roastery & Co.** You're tidying up stale objects and run a `DROP SCHEMA` — but on the wrong one. It held a live table a team depends on. As the admin, you're the person who's expected to recover it *and* to set the retention policy that made recovery possible in the first place.

We'll cover:
1. What Time Travel is and why it's your continuous data-protection layer
2. Standing up a schema to manage (with a table inside)
3. The incident — an accidental `DROP SCHEMA`
4. Recovering the entire schema in one statement with `UNDROP SCHEMA`
5. Your angle — tuning `DATA_RETENTION_TIME_IN_DAYS` and the Time Travel vs Fail-safe boundary

:books: **By the end you'll be able to:**
- Explain the Time Travel retention window and what it covers
- Recover a dropped schema (and everything in it) with `UNDROP SCHEMA`
- Inspect and set `DATA_RETENTION_TIME_IN_DAYS` on an object
- Explain the difference between self-service Time Travel and Snowflake-managed Fail-safe
- Reason about the cost/recoverability tradeoff of longer retention
- Use CoCo to diagnose and recover from an accidental drop

**Time:** ~15 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set Your Session Context

Every Snowflake session runs under a role and a warehouse. The cell below sets your everyday role and makes sure your personal warehouse and database exist. Note that this lab works in its **own** schema, `TT_ADMIN_LAB` — one you can safely drop and recover without touching anything else in your database.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
CREATE DATABASE IF NOT EXISTS {{my_db}};


---
## 1 · What Time Travel Gives You

[Time Travel](https://docs.snowflake.com/en/user-guide/data-time-travel) is Snowflake's built-in, continuous data-protection layer. For a set **retention period**, Snowflake keeps the historical versions of every table, schema, and database — so an accidental drop, a bad delete, or a wrong overwrite is recoverable without a backup restore. As the admin, this is the capability you configure, monitor, and are expected to wield when something goes wrong.

Within the retention window you (or the object owner) can:

| Do this | With |
|---|---|
| **Bring back** a dropped table, schema, or database | `UNDROP` |
| **Query** an object as it was *just before* a statement | `BEFORE(statement => '<query_id>')` |
| **Query** an object as it was at a point in time | `AT(timestamp => ...)` / `AT(offset => -seconds)` |
| **Recover** an older state without duplicating storage | `CREATE ... CLONE ... AT/BEFORE(...)` |

**The retention window is a policy you set:**

| Edition | Default | Maximum |
|---|---|---|
| Standard | 1 day | 1 day |
| Enterprise and above | 1 day | 90 days (per object, via `DATA_RETENTION_TIME_IN_DAYS`) |

> :bulb: **Time Travel vs. Fail-safe — an admin must know the line.** Time Travel is **self-service** recovery within the retention window you configure. After that window closes, data enters **Fail-safe** — a short, fixed **7-day**, **non-configurable** period recoverable *only by Snowflake Support*. Coverage depends on table type: **permanent** tables get both Time Travel (up to 90 days on Enterprise+) and Fail-safe; **transient** and **temporary** tables get a maximum 1-day Time Travel window and **no Fail-safe**. You design recovery around Time Travel; Fail-safe is a last resort, not a plan.

---
## 2 · Stand Up a Schema to Manage

First, create the schema you'll manage and put a real table in it — so the drop has something to lose. This copies `ORDERS` from the read-only shared source into a table you own inside `TT_ADMIN_LAB`.

In [ ]:
CREATE SCHEMA IF NOT EXISTS {{my_db}}.TT_ADMIN_LAB;

CREATE OR REPLACE TABLE {{my_db}}.TT_ADMIN_LAB.ORDERS_MANAGED AS
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS;

SELECT COUNT(*) AS rows_in_managed_table FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_MANAGED;


> :mag_right: **Review the output.** The `TT_ADMIN_LAB` schema now holds `ORDERS_MANAGED` with a healthy row count — this is the state you'll lose, then recover in full.

In [ ]:
-- Confirm the schema exists before the incident
SHOW SCHEMAS LIKE 'TT_ADMIN_LAB' IN DATABASE {{my_db}};


> :mag_right: **Review the output.** One row — `TT_ADMIN_LAB` is present. Keep this in mind; you'll run the same `SHOW` after the drop and after the recovery to prove each step.

---
## 3 · The Incident — An Accidental `DROP SCHEMA`

Here's the mistake every admin dreads. Mid-cleanup, you [`DROP SCHEMA`](https://docs.snowflake.com/en/sql-reference/sql/drop-schema) — but on the live one. The schema and **everything inside it** disappear in one statement.

Drop it and try to read a table inside it — you'll see the failure. Then, in Section 4, recover the whole schema in one statement.

In [ ]:
-- Oops: dropped the wrong schema during cleanup — the whole thing is gone
DROP SCHEMA {{my_db}}.TT_ADMIN_LAB;

-- Try to read a table inside it — this WILL fail
SELECT COUNT(*) FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_MANAGED;


> ⛔ **Expected error: schema does not exist.** Reading `ORDERS_MANAGED` fails because `TT_ADMIN_LAB` and everything inside it are gone — that's what a dropped schema looks like to anyone querying it. **Don't reach for a rebuild.** Within the retention window, this is a one-statement recovery — you'll do it in Section 4.

---
## 4 · Recover With `UNDROP SCHEMA`

[`UNDROP`](https://docs.snowflake.com/en/sql-reference/sql/undrop-schema) restores the most recent dropped version of an object — and it works at **schema and database** scope, not just tables. One statement brings back the schema *and every object inside it*, exactly as it was.

> :point_right: **Your turn — fill in the blank.** The whole `TT_ADMIN_LAB` schema is gone. Which single command restores the most recent dropped version of an object — and works at **table, schema, and database** scope?

In [ ]:
___ SCHEMA {{my_db}}.TT_ADMIN_LAB;   -- one-statement recovery of a dropped schema

SHOW SCHEMAS LIKE 'TT_ADMIN_LAB' IN DATABASE {{my_db}};


<details>
<summary>📖 Reference solution</summary>

    UNDROP SCHEMA {{my_db}}.TT_ADMIN_LAB;

`UNDROP` reverses a drop within the retention window. At schema scope it brings back the schema *and every object inside it* in one statement.
</details>

> :mag_right: **Review the output.** The schema is back — one row again. Now confirm its contents survived the round trip.

In [ ]:
-- The table inside the schema came back with it, rows intact
SELECT COUNT(*) AS rows_after_undrop FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_MANAGED;


> :mag_right: **Review the output.** `rows_after_undrop` matches the original count — the schema *and* its table are fully restored. `UNDROP SCHEMA` recovered the whole container in one move.

> 🧑‍💻 **Why this matters to you as a platform admin.** You're the one who gets the 2 AM call when something vanishes. `UNDROP SCHEMA` is your fastest way back — one statement restores the whole schema, grants and all: no rebuild script, no restore-from-backup ticket, no re-coordinating access with every team. It comes back exactly as it was.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Show me any recently dropped schemas or tables in {{my_db}} so I can see what's recoverable with UNDROP.</pre>

In real life you might not know *what* was dropped — CoCo can run the `SHOW ... HISTORY` commands to surface dropped objects and their `dropped_on` timestamps, so you know what's still inside the retention window.

### See the whole story in your query history. 🧾

You just dropped and restored an entire schema — and every statement you ran is recorded. `INFORMATION_SCHEMA.QUERY_HISTORY` lets you read your own recent activity back as a mini **audit trail**. Query it for anything that touched `TT_ADMIN_LAB` and you'll see the recovery in order: the `DROP SCHEMA`, the `UNDROP SCHEMA`, and the checks around them — newest first.

In [ ]:
-- The audit trail: every recent statement that touched the TT_ADMIN_LAB schema, newest first.
-- INFORMATION_SCHEMA.QUERY_HISTORY is a table function — wrap it in TABLE(...).
SELECT QUERY_ID, QUERY_TYPE, EXECUTION_STATUS, START_TIME, QUERY_TEXT
FROM TABLE({{my_db}}.INFORMATION_SCHEMA.QUERY_HISTORY(RESULT_LIMIT => 200))
WHERE QUERY_TEXT ILIKE '%TT_ADMIN_LAB%'
ORDER BY START_TIME DESC;


> :mag_right: **Review the output.** Read top-to-bottom (newest first) and you can retrace the recovery. Your `DROP SCHEMA` shows as `QUERY_TYPE = DROP` — and here's the neat part: your `UNDROP SCHEMA` shows as **`RESTORE`**, Snowflake's internal name for the operation. Reconstructing *what happened* is the first move in any recovery, and `QUERY_HISTORY` is where you do it.

> :bulb: **Two lenses on history.** `INFORMATION_SCHEMA.QUERY_HISTORY` returns *your* recent queries (last 7 days). As an admin you'll more often reach for the account-wide [`SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/query_history) view, which spans every user — same idea, wider lens, longer retention.

### 📞 The call: recover deleted rows from a *live* table.

You've barely finished your coffee when an analyst pings you, frantic: *"I ran a `DELETE` on the production orders table and the `WHERE` was wrong — I wiped a whole store's orders. And it's a **live** table, orders are still landing in it right now!"*

This is a different recovery from the schema `UNDROP` you just did. You **can't** simply restore the whole table as it was — new, legitimate orders have arrived *since* the bad `DELETE`, and a wholesale restore would throw those away. The admin move is **surgical**: restore a *copy* of the table as of just before the delete, find the rows now missing from live production (matched by a unique `ORDER_ID`), and re-insert **only those**. The live orders are never touched.

Let's stand up a production table and reproduce the call.

In [ ]:
-- The analyst's production table (inside the schema you just recovered), with a unique ORDER_ID per row.
CREATE OR REPLACE TABLE {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD AS
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS;

SELECT COUNT(*) AS rows_in_prod FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD;


In [ ]:
-- 1) The analyst's mistake: a wrong WHERE wipes every order for store 3.
DELETE FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD WHERE store_id = 3;
SET bad_delete = LAST_QUERY_ID();   -- capture the exact statement so we can look back to just before it

-- 2) Meanwhile, production keeps taking new orders (brand-new ORDER_IDs). These must survive the recovery.
INSERT INTO {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD (order_id, store_id, customer_id, order_ts, total_amount) VALUES
  (900001, 7, 4242, CURRENT_TIMESTAMP(), 12.50),
  (900002, 9, 4243, CURRENT_TIMESTAMP(), 8.75);

SELECT COUNT(*) AS rows_live_now FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD;


> :mag_right: **Review the output.** Production dropped store 3's orders, then gained two brand-new ones — `rows_live_now` reflects both changes at once. That's the trap: the table is a moving target. If you restored the whole pre-delete snapshot over the top, you'd erase orders `900001` and `900002` that arrived *after* the mistake. You need to put back only what's missing.

> 📌 **Lab shortcut vs. real life.** You captured `$bad_delete` in the very cell that caused it — tidy for a lab, but not how it happens in production. Normally you'd discover the damage later and trace the offending statement in [Query History](https://docs.snowflake.com/en/user-guide/ui-snowsight-activity#query-history) (filter by user, time window, and statement type). CoCo can run that search for you:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Search my recent query history for DELETE statements against {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD and return the query_id and timestamp of the most recent one.</pre>

That's how you'd find the offending statement for real — CoCo reads `QUERY_HISTORY` and hands you the anchor for `BEFORE(statement => ...)`.

> :point_right: **Your turn — fill in the blank.** First, restore a **copy** of the table as it was *just before* the bad `DELETE` (a zero-copy clone). Which Time Travel clause anchors to the statement you captured in `$bad_delete`?

In [ ]:
-- Zero-copy clone of the pre-delete state into a separate snapshot table you can compare against.
CREATE OR REPLACE TABLE {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD_SNAPSHOT
  CLONE {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD ___(statement => $bad_delete);   -- look-back clause

SELECT COUNT(*) AS rows_in_snapshot FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD_SNAPSHOT;


<details>
<summary>📖 Reference solution</summary>

    CREATE OR REPLACE TABLE {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD_SNAPSHOT
      CLONE {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD BEFORE(statement => $bad_delete);

`BEFORE(statement => $bad_delete)` clones the table exactly as it was the instant before the bad `DELETE` ran — deleted rows included, and none of the orders that arrived afterward.
</details>

> :point_right: **Trust, but verify — before you write.** You're about to `INSERT` the snapshot rows that are missing from live production. First *prove* that set is exactly what you expect: the wrongly-deleted store-3 orders, and nothing else. Run the check below **before** the insert — never run a recovery `INSERT` you haven't eyeballed.

In [ ]:
-- Preview EXACTLY what the reconcile INSERT will add: snapshot rows whose ORDER_ID is not in live PROD.
-- Confirm the count matches the store-3 orders you lost, and that every row is store 3 — before you insert.
SELECT COUNT(*) AS rows_to_restore,
       COUNT(DISTINCT s.store_id) AS distinct_stores,
       MIN(s.store_id) AS sample_store_id
FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD_SNAPSHOT s
WHERE NOT EXISTS (
  SELECT 1 FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD p
  WHERE p.order_id = s.order_id
);


> :mag_right: **Review the output.** `rows_to_restore` should equal the number of store-3 orders the bad `DELETE` wiped, `distinct_stores` should be **1**, and `sample_store_id` should be **3** — proof the set is precisely the deleted rows and nothing legitimate. Only now is it safe to run the `INSERT`.

In [ ]:
-- Re-insert ONLY the rows missing from live production: rows in the snapshot whose ORDER_ID isn't
-- in ORDERS_PROD now — i.e. the wrongly-deleted store-3 orders. The new orders that arrived after the
-- delete live only in PROD (not the snapshot), so they are left completely untouched.
INSERT INTO {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD
SELECT s.*
FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD_SNAPSHOT s
WHERE NOT EXISTS (
  SELECT 1 FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD p
  WHERE p.order_id = s.order_id
);

SELECT COUNT(*) AS rows_after_reconcile FROM {{my_db}}.TT_ADMIN_LAB.ORDERS_PROD;


> :mag_right: **Review the output.** `rows_after_reconcile` is back up to the original count **plus** the two new live orders — store 3's rows are restored *and* `900001`/`900002` are still there. You recovered the deleted data without rolling back a single legitimate order. That's the pattern for any table still being written to: **clone `BEFORE` the incident, match on a unique key, insert only what's missing** — never a blind overwrite.

> :bulb: **Why not `UNDROP` or `SWAP`?** `UNDROP` brings back a *dropped object*, not rows deleted from a live one. A whole-table `SWAP`/overwrite would clobber the orders that arrived after the delete. Matching on the **unique `ORDER_ID`** is what makes the recovery surgical and safe on a live table.

---
## 5 · Your Angle — Platform Admin: Govern the Retention Window

Recovery only worked because the schema was inside its retention window. Setting that window is an **admin policy decision** — longer retention means more recoverability, but Time Travel storage of changed/deleted data counts toward your storage costs. You balance the two.

`DATA_RETENTION_TIME_IN_DAYS` can be set at account, database, schema, or table level (the lowest, most specific level wins). Let's inspect it, then set it explicitly on the recovered schema.

In [ ]:
-- Account default — the value your schema inherits unless overridden
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN ACCOUNT;


In [ ]:
-- Your database
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN DATABASE {{my_db}};


In [ ]:
-- What retention is in effect on the schema right now?
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN SCHEMA {{my_db}}.TT_ADMIN_LAB;


> :mag_right: **Review the three outputs (account → database → schema).** Each shows `value` = `1` with a **blank `level`** — the schema inherits the account default; nothing is pinned lower down yet. The `level` column is the tell: blank = inherited, `DATABASE`/`SCHEMA`/`TABLE` = set explicitly at that scope. On Standard Edition the maximum is 1 day; Enterprise Edition and above allow up to 90. You'll pin it explicitly next.

### Set retention explicitly. 🛡️

Below you raise retention on the schema to **7 days** — longer than the inherited account default, so you can watch the value visibly change. (7 requires Enterprise Edition or above; Standard Edition caps at 1.) You'd set a longer window like this for schemas holding data you may need to recover further back. The tradeoff: more days = more recoverability = more Time Travel storage.

> :point_right: **Your turn — fill in the blank.** Pin the retention window on this schema explicitly. Which object parameter sets the number of days of Time Travel history Snowflake keeps?

In [ ]:
-- 7 days (Enterprise+; Standard Edition caps at 1) — longer than the inherited account default
ALTER SCHEMA {{my_db}}.TT_ADMIN_LAB SET ___ = 7;   -- the Time Travel retention parameter

SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN SCHEMA {{my_db}}.TT_ADMIN_LAB;


<details>
<summary>📖 Reference solution</summary>

    ALTER SCHEMA {{my_db}}.TT_ADMIN_LAB SET DATA_RETENTION_TIME_IN_DAYS = 7;

`DATA_RETENTION_TIME_IN_DAYS` is the Time Travel retention dial — settable at account, database, schema, or table level (the lowest, most specific level wins).
</details>

> :mag_right: **Review the output.** The `value` is now **`7`** and the `level` shows `SCHEMA` — the setting is pinned to this schema explicitly rather than inherited from the account's `1`. New tables created here inherit the `7`-day window unless they override it themselves.

> :bulb: **Governance rule of thumb.** Set generous retention on schemas holding hard-to-reconstruct source or curated data; keep it short on cheap-to-rebuild staging. Retention is a per-object dial, so you can be deliberate rather than blanket.

> 📌 **Clock note.** "1 day" is a *rolling* 24-hour window measured from each change — not a calendar-day boundary. A drop at 3 PM today stays recoverable until 3 PM tomorrow, then the history ages out.

> :point_right: **Your turn — write this one yourself.** You set retention on the *schema*. Retention resolves at account → database → schema → **table**, with the most specific level winning. Write an `ALTER TABLE` that pins `DATA_RETENTION_TIME_IN_DAYS = 3` directly on `ORDERS_MANAGED` (a different value than the schema's 7, so the override is obvious), then `SHOW PARAMETERS` to confirm the setting now lives at table level.

In [ ]:
-- Your turn: pin DATA_RETENTION_TIME_IN_DAYS = 3 on the ORDERS_MANAGED table (a different value than the schema's 7 — lowest, most-specific level wins), then SHOW PARAMETERS to confirm


<details>
<summary>📖 Reference solution</summary>

    ALTER TABLE {{my_db}}.TT_ADMIN_LAB.ORDERS_MANAGED SET DATA_RETENTION_TIME_IN_DAYS = 3;

    SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN TABLE {{my_db}}.TT_ADMIN_LAB.ORDERS_MANAGED;

A table-level setting overrides the schema it inherits from — `value` now shows `3` (not the schema's `7`) and `level` shows `TABLE`. The lowest, most specific level wins.
</details>

---
## :robot_face: CoCo Assist

For an admin, CoCo is a policy and recovery partner — it can recover objects, explain retention math, and reason about the cost tradeoff. Here's how you'd use it in the moment.

> :warning: **How to get the most from this section:**
>
> 1. **Read CoCo's response carefully** — the explanation is where the learning happens.
> 2. **CoCo is interactive** — ask follow-ups until it clicks ("Does UNDROP restore grants too?").
> 3. **The prompts below are starting points** — your real learning comes from trying your own.

---

**Example 1: Understand the recovery scopes**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Explain the scopes UNDROP works at (table, schema, database) and its limits, with reference to the different table types — does it restore the most recent version only? What happens to grants and dependent objects when I UNDROP a schema?</pre>

---

**Example 2: Reason about the retention / cost tradeoff**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">I manage schemas with different recovery needs. How does DATA_RETENTION_TIME_IN_DAYS affect storage cost, and how does Time Travel differ from Fail-safe for planning recovery? Recommend retention values for source, curated, and staging schemas.</pre>

---

> :bulb: **These are just examples.** CoCo can help with any admin recovery question: "Which objects were dropped in the last day?", "How do I set account-level default retention?", "What's still recoverable if the Time Travel window has passed?" Ask in plain English and follow up until you're confident.

---
## Verify Your Work

The check below confirms the dropped schema was recovered, its table survived with all rows, and an explicit retention policy is in place.

In [ ]:
all_passed = True

# Schema recovered
try:
    n = len(session.sql(f"SHOW SCHEMAS LIKE 'TT_ADMIN_LAB' IN DATABASE {my_db}").collect())
    ok = n >= 1
    print(('✅' if ok else '⛔') + f' TT_ADMIN_LAB schema recovered: {"present" if ok else "missing"}')
    if not ok:
        all_passed = False
        print('   Re-run Section 4 — UNDROP SCHEMA should bring it back.')
except Exception as e:
    print(f'⛔ Schema check ERROR — {e}')
    all_passed = False

# Table inside the schema came back with all rows
try:
    src = session.sql('SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS').collect()[0][0]
    got = session.sql(f'SELECT COUNT(*) FROM {my_db}.TT_ADMIN_LAB.ORDERS_MANAGED').collect()[0][0]
    ok = got == src
    print(('✅' if ok else '⛔') + f' ORDERS_MANAGED restored: {got} rows (source has {src})')
    if not ok:
        all_passed = False
except Exception as e:
    print(f'⛔ Table check ERROR — {e}')
    all_passed = False

# Live-table recovery: deleted rows restored AND orders that arrived after the delete preserved
try:
    src3 = session.sql('SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS WHERE store_id = 3').collect()[0][0]
    prod3 = session.sql(f'SELECT COUNT(*) FROM {my_db}.TT_ADMIN_LAB.ORDERS_PROD WHERE store_id = 3').collect()[0][0]
    live = session.sql(f'SELECT COUNT(*) FROM {my_db}.TT_ADMIN_LAB.ORDERS_PROD WHERE order_id IN (900001, 900002)').collect()[0][0]
    ok = (prod3 == src3) and (live == 2)
    print(('✅' if ok else '⛔') + f' ORDERS_PROD reconciled: {prod3}/{src3} store-3 rows restored, {live}/2 live orders preserved')
    if not ok:
        all_passed = False
        print('   Re-run the live-recovery subsection — clone BEFORE the delete, then INSERT the missing rows.')
except Exception as e:
    print(f'⛔ Live-recovery check ERROR — {e}')
    all_passed = False

if all_passed:
    print('\n🎉 All checks passed — you recovered a dropped schema, reconciled deleted rows on a live table, and set its retention policy.')


---
### Check your understanding. 🧠
A few questions on this lab's key themes. Run each quiz cell below, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q4")

---
### :snowflake: Key Takeaways

- Time Travel retains history for a **retention period** (1 day default; up to 90 on Enterprise+) — your continuous, self-service data-protection layer
- **`UNDROP`** works at table, schema, and database scope — `UNDROP SCHEMA` restores a dropped schema *and every object inside it* in one statement
- For rows deleted from a **live** table, recover *surgically*: `CLONE ... BEFORE(statement => …)` the pre-delete state, then re-insert only the rows missing by **unique id** — never a blind overwrite that would clobber data written since the incident
- **`DATA_RETENTION_TIME_IN_DAYS`** sets the window at account/database/schema/table level (most specific wins) — a per-object policy dial
- Longer retention = more recoverability but more Time Travel **storage cost** — set it generously on hard-to-rebuild data, short on cheap staging
- **Time Travel is self-service; Fail-safe is Snowflake-managed** and non-configurable — design recovery around Time Travel, treat Fail-safe as last resort
- **CoCo is your recovery partner** — it can recover objects and reason about the retention/cost tradeoff

**The Other Personas:**
- The **Data Analyst** notebook recovers a curated analysis table from a bad **`DELETE`**
- The **Data Engineer** notebook recovers a staging table from a bad pipeline **overwrite**

> :bulb: Each persona lab is independent and uses its own objects — you can run them in any order.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.